# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print basic description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps identify the structure of the Croissant dataset.

The `@id` of each record set, field, and column will be used for referencing.

Let's enumerate all available record sets and their fields. (If you receive empty results, the dataset record sets may be embedded in the Croissant schema - in that case, you can explore `dataset.record_sets`.)

In [ ]:
# List all record sets by their @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    # List fields within each record set
    fields = rs.get('field', [])
    print("  Fields:")
    for field in fields:
        print(f"    Field @id: {field['@id']}, name: {field.get('name', 'N/A')}")
    print()
# Example record listing from first record set
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"Showing first 3 records from {first_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        if i >= 3:
            break
        print(rec)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record sets and their fields are referenced by their `@id` values.

We will dynamically extract the data from each record set, storing them in a dictionary keyed by each record set's `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Dataframe for record set {rs_id} has columns:")
    print(df.columns.tolist())
    print(df.head(3), "\n")

# Choose one record set for deeper analysis (e.g., the first one)
chosen_record_set_id = record_sets[0]['@id'] if record_sets else None
if chosen_record_set_id:
    print(f"Columns in chosen record set ({chosen_record_set_id}):")
    print(dataframes[chosen_record_set_id].columns.tolist())
    dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All references to fields are by their `@id` values.

In [ ]:
# EDA: Remove outliers, normalize, and group
if chosen_record_set_id:
    df = dataframes[chosen_record_set_id]
    # Identify numeric fields by @id
    numeric_fields = []
    for field in record_sets[0].get('field', []):
        # Suppose we identify numeric fields by dataType
        if field.get('dataType') in ('Float', 'Integer', 'Number'):
            numeric_fields.append(field['@id'])

    print("Numeric fields:", numeric_fields)

    # Example: Filter on first numeric field if exists
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        if numeric_field_id in df.columns:
            threshold = 10
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())
            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        else:
            print("Numeric field not in DataFrame columns.")
    # Attempt grouping by a non-numeric field
    group_fields = [f['@id'] for f in record_sets[0].get('field', []) if f.get('dataType') == 'Text']
    print("Group fields:", group_fields)
    if group_fields:
        group_field_id = group_fields[0]
        if group_field_id in df.columns:
            grouped_df = df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No appropriate group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of a numeric field and group averages.

In [ ]:
# Visualization of EDA results
if chosen_record_set_id:
    df = dataframes[chosen_record_set_id]
    # Plot histogram for first numeric field
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        if numeric_field_id in df.columns:
            plt.figure(figsize=(6,4))
            df[numeric_field_id].dropna().hist(bins=10)
            plt.title(f"Distribution of {numeric_field_id}")
            plt.xlabel(numeric_field_id)
            plt.ylabel("Frequency")
            plt.show()
    # If grouping was performed
    if group_fields:
        group_field_id = group_fields[0]
        if group_field_id in df.columns and numeric_field_id in df.columns:
            grouped = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
            grouped.plot(kind='bar', figsize=(8,4))
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xlabel(group_field_id)
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we loaded and explored the clinical genotype-phenotype dataset for NC-21OHD-associated infertility.
- Data was programmatically accessed by `@id`, ensuring reproducible referencing of record sets, fields, and columns.
- Per the `@id` schema, we performed EDA, filtered and normalized numeric variables, and visualized their distributions.
- This approach allows for robust exploration and downstream analytics for small-scale, high-quality biomedical datasets.